In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/new_notebooks/us-birth/small_bench/checkpoints/pre_cell_19.pickle

In [ ]:
%%cudf.pandas.profile
### cell 19 ###

# Vectorized GPU datetime index conversion using astype casting
months = births_by_date.index.get_level_values(0)
days = births_by_date.index.get_level_values(1)
# Cumulative days before each month in 2020 (leap year)
month_offsets = pd.Series([0, 31, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335], dtype="int32")
# Compute day-of-year (0-based)
day_of_year = month_offsets.take(months - 1) + days - 1
# Days from 1970-01-01 to 2020-01-01 = 18262
epoch_days = 18262
# Nanoseconds per day
ns_per_day = 86400 * 10**9
# Build nanosecond timestamps and cast to datetime64[ns] on GPU
ts_ns = (epoch_days + day_of_year) * ns_per_day
births_by_date.index = ts_ns.astype("datetime64[ns]")
births_by_date.head()